# 🎵 AnimalMind — Audio Vocalization Classifier Training Pipeline

Fine-tunes **Wav2Vec2** on pet vocalizations (`bark`, `meow`, `whine`, `growl`, `hiss`, `silence`):
- **Augmentations**: SpecAugment, White Noise Injection, Pitch Shifting, Time Stretching.
- **Uncertainty Calibration**: Temperature Scaling ($T$) to minimize Expected Calibration Error (ECE).
- **Hub Export**: Pushes model directly to Hugging Face Hub (`firstoff/animalmind-audio-classifier`).

In [2]:
# 1. Verify GPU Acceleration
!nvidia-smi

Wed Jul 29 00:13:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# 2. Clone Repository & Install Training Dependencies
!git clone https://github.com/firstoff23/AnimalMind.git
%cd AnimalMind/ml_backend
!pip install -q -r requirements_training.txt

Cloning into 'AnimalMind'...
remote: Enumerating objects: 47489, done.
remote: Counting objects: 100% (318/318), done.
remote: Compressing objects: 100% (164/164), done.
remote: Total 47489 (delta 201), reused 238 (delta 143), pack-reused 47171 (from 2)
Receiving objects: 100% (47489/47489), 100.78 MiB | 18.99 MiB/s, done.
Resolving deltas: 100% (17922/17922), done.
/content/AnimalMind/ml_backend


In [4]:
# 3. Configure Hugging Face Secret Token (Optional)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("🔑 Hugging Face Token loaded.")
except Exception:
    print("ℹ️ No HF_TOKEN secret found.")

🔑 Hugging Face Token loaded.


In [5]:
# 4. Launch 20-Epoch Audio Classifier Fine-Tuning Pipeline
!python -m training.train_audio_classifier \
    --epochs 20 \
    --batch-size 16 \
    --lr 1e-4 \
    --model-name facebook/wav2vec2-base \
    --output-dir models/animalmind-audio-classifier \
    --push-to-hub firstoff/animalmind-audio-classifier

[AudioTraining] Initializing Audio Classification Fine-Tuning (facebook/wav2vec2-base)...
Classes: ['bark', 'meow', 'whine', 'growl', 'hiss', 'silence']
preprocessor_config.json: 100% 159/159 [00:00<00:00, 723kB/s]
config.json: 100% 1.84k/1.84k [00:00<00:00, 652kB/s]

pytorch_model.bin: downloading bytes:  21% 81.0M/380M [00:01<00:04, 73.2MB/s, 5.29MB/s  ]
pytorch_model.bin: downloading bytes:  28% 108M/380M [00:02<00:02, 103MB/s, 7.69MB/s  ]  
pytorch_model.bin: downloading bytes:  49% 184M/380M [00:02<00:01, 151MB/s, 14.2MB/s  ]
pytorch_model.bin: reconstructing file:  22% 83.7M/380M [00:02<00:09, 32.3MB/s, 6.95MB/s  ]
pytorch_model.bin: downloading bytes:  60% 228M/380M [00:02<00:00, 169MB/s, 19.0MB/s  ]
pytorch_model.bin: downloading bytes:  67% 255M/380M [00:02<00:00, 158MB/s, 22.3MB/s  ]
pytorch_model.bin: reconstructing file:  67% 256M/380M [00:02<00:00, 205MB/s, 12.7MB/s  ] 
pytorch_model.bin: reconstructing file:  80% 304M/380M [00:03<00:00, 214MB/s, 23.4MB/s  ]
pytorch_model.

In [6]:
# 5. Inspect Calibration & Training Summary
import json, torch
with open("training/audio_training_metrics.json", "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"🌡️ Calibrated Temperature T : {data.get('calibrated_temperature', 1.0):.4f}")
print(f"📊 ECE : {data.get('ece', 0.0):.4f}")

🌡️ Calibrated Temperature T : 1.5000
📊 ECE : 0.0275
